# 4. Fine-Tune Model
This stage aims to retrain (fine-tune) the model that was previously developed in stage 2. The process uses an annotated/labeled dataset that is either different from or completely new compared to the one used for the initial training. <br>

After fine-tuning, the model is expected to achieve better performance than the previous version. However, performance may also decrease if the resolution or quality of the new dataset differs significantly from the original training data. <br>

Therefore, the fine-tuning step is considered **optional**. <br>


In [5]:
!pip install ultralytics
!pip install roboflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 81.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.0/250.0 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 78.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 127.8 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: idna 3.18
    Uninstalling idna-3.18:
      Successfully uninstalled idna-3.18


In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
import torch

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using {DEVICE}')

Using cuda


In [8]:
from roboflow import Roboflow
rf = Roboflow(api_key="u3UIOc5ovdRCfiZLGubI")
project = rf.workspace("mwahyur").project("oilpalm-tree-detection-vllsd")
version = project.version(2)
dataset = version.download("yolo26")


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Oilpalm-Tree-Detection-2 in yolo26:: 100%|██████████| 235/235 [00:00<00:00, 6391.93it/s]


In [9]:
from ultralytics import YOLO

# Load the trained model
model = YOLO(r'/content/drive/MyDrive/PROJECT/oilpalm-tree-detection-yolov26/model/oilpalm_detection_model_yolov26n.pt')

# Path to the new data.yaml file
data_yaml_path = r'/content/Oilpalm-Tree-Detection-2/data.yaml' # External anotated/labeled data downloaded from Roboflow, but this is example from previous or current version of the dataset

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


## Run Fine-Tune
Add additional parameters in the model fine-tuning process that were not previously used in the initial training.

In [10]:
# Fine-tune the model with new data
model.train(
    data=data_yaml_path,
    epochs=5,  # Number of epochs
    imgsz=640,  # Input image size
    batch=16,  # Batch size
    lr0=0.001,  # Learning rate
    optimizer='Adam',  # Optimizer
    weight_decay=0.0005,  # Regularization
    momentum=0.9,  # Momentum
    patience=5,  # Patience for early stopping
    augment=False  # Data augmentation (`True`: data augmentation is applied, `False`: No augmentation)
)

Ultralytics 8.4.70 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Oilpalm-Tree-Detection-2/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=5, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/PROJECT/oilpalm-tree-detection-yolov26/model/oilpalm_detection_model_yolov26n.pt, momentum=0.9, mosaic=1.0, multi_scale=0.0, na

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x79f6bc145ac0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

## Save the Fine-Tune Model

In [11]:
# Save the fine-tuned model
fine_tuned_model_path = r'/content/drive/MyDrive/PROJECT/oilpalm-tree-detection-yolov26/model/oilpalm_detection_model_yolov26n_fine_tune.pt'
model.save(fine_tuned_model_path)

print(f"Fine-tuned model has been saved to {fine_tuned_model_path}.")

Fine-tuned model has been saved to /content/drive/MyDrive/PROJECT/oilpalm-tree-detection-yolov26/model/oilpalm_detection_model_yolov26n_fine_tune.pt.
